In [0]:
# DAC043: complete CT chest / chest-abdomen / chest-abdomen-pelvis /
# abdomen-pelvis contrast cohort, 2022-01-01 through 2024-12-31 inclusive.
# Staged in CC; Ben replaces /Workspace/Shared/ADC-DB/Projects/DAC043.
# Run All writes the same two project tables and schema view as the original.
#
# Eligibility uses performed PACS examinations and recorded contrast protocols.
# "With contrast" routine protocols are an IV proxy, not proof of administration.
# Oral-only / explicitly non-IV studies do not qualify. Generic protocol names
# qualify only when a linked same-family contrast event or report technique
# supplies positive evidence. No sample cap or report-completeness filter.
# No Millennium report-date fallback: that is commonly the authorisation date.

In [0]:
import builtins
import re

project_identifier = "dac043"
TARGET_CATALOG = "5_projects"
TARGET = f"{TARGET_CATALOG}.{project_identifier}"
PSEUDONYM_SALT = "5055d702c374f9dfd3e48cac740797bdb9d2ce7c94994be6ec601a6e6006cd6b"
MIN_EXAM_DATE = "2022-01-01"
MAX_EXAM_DATE_EXCLUSIVE = "2025-01-01"
PIPELINE_VERSION = "dac043.v2.0"

# Current and historical NICIP aliases checked against the landed TRUD dictionary.
# The four main local contrast codes are CCHESC / CCABDC / CCHAPC / CABPEC.
EVENT_FAMILIES = {
    6182886: ("CHEST", False), 6183094: ("CHEST", True),
    6184187: ("CA", False), 6180914: ("CA", True),
    6184208: ("CAP", False), 6183849: ("CAP", True),
    6180036: ("AP", False), 6182898: ("AP", True),
    6182338: ("CA", True),  # Thorax without contrast + abdomen with contrast.
}
EVENT_CD_SQL = ",".join(str(k) for k in builtins.sorted(EVENT_FAMILIES))
EVENT_FAMILY_CASE = " ".join(
    f"WHEN {k} THEN '{v[0]}'" for k, v in EVENT_FAMILIES.items()
)
EVENT_CONTRAST_CASE = " ".join(
    f"WHEN {k} THEN {str(v[1]).lower()}" for k, v in EVENT_FAMILIES.items()
)

def sql_literal(value):
    return "'" + value.replace("'", "''") + "'"

def family_sql(text_column):
    # Text has already been lowercased and punctuation converted to spaces.
    chest = f"{text_column} RLIKE r'(^| )(chest|thorax|thoracic|thor|thoraco)( |$)'"
    abdo = f"{text_column} RLIKE r'(^| )(abdomen|abdominal|abdo|abd|abdomino)( |$)'"
    pelvis = f"{text_column} RLIKE r'(^| )(pelvis|pelvic|pelv|pelvi)( |$)'"
    cap = f"{text_column} RLIKE r'(^| )(cap|tap)( |$)'"
    excluded = (
        r"(^| )(head|brain|neck|nk|spine|spinal|cervical|lumbar|aorta|aortic|"
        r"angio|angiogram|angiography|angiographic|ctpa|cta|venogram|venography|vascular|"
        r"biopsy|drainage|aspiration|guided|intervention|interventional|planning|"
        r"radiotherapy|brachytherapy|colonography|colon|kub|urogram|urography|"
        r"hrct|high resolution|hi res|low dose|summit|pet|spect|scapula|scaphoid)( |$)"
    )
    return f"""CASE
      WHEN {text_column} RLIKE r'{excluded}' THEN NULL
      WHEN ({chest} AND {abdo} AND {pelvis}) OR {cap} THEN 'CAP'
      WHEN {chest} AND {abdo} AND NOT ({pelvis}) THEN 'CA'
      WHEN {abdo} AND {pelvis} AND NOT ({chest}) THEN 'AP'
      WHEN {chest} AND NOT ({abdo}) AND NOT ({pelvis}) THEN 'CHEST'
    END"""

# Avoid "with AND without contrast" being mistaken for a noncontrast-only study.
# A mixed thorax-without / abdomen-with contrast protocol still uses IV contrast.
NO_IV_PATTERN = (
    r"(^| )(no|without|non|wo|w o) +(iv |intravenous |intra venous )?"
    r"(contrast|cont|con)( |$)|(^| )(plain|nc|noncontrast|unenhanced)( |$)|"
    r"(^| )(no|without|non) +(iv|intravenous)( |$)|"
    r"(^| )(iv|intravenous) +(contrast )?(not given|not administered|withheld)( |$)"
)
IV_PATTERN = r"(^| )(iv|i v|intravenous|intra venous|ivc|pivc|civ)( |$)"
IV_CONTRAST_TECHNIQUE_PATTERN = (
    r"(^| )(iv|i v|intravenous|intra venous) +(administration of +)?"
    r"(iodinated +)?((and )?(oral|rectal) +)?contrast( |$)|"
    r"(^| )contrast +(was +)?(given|administered|injected) +intravenously( |$)"
)
CONTRAST_PATTERN = r"(^| )(contrast|cont|con|contras|postcontrast|cect)( |$)|[+] *c( |$)|withcontrast"
TECHNIQUE_PATTERN = r"(?i)(?:^|[\r\n])[ \t]*technique[ \t]*[:\-]?[ \t]*([^\r\n]*)"

COHORT_CTES = rf"""
pacs_normalized AS (
  SELECT p.*,
    trim(regexp_replace(
      regexp_replace(lower(COALESCE(p.EXAMINATION_DESCRIPTION, '')), '^ct(?=abd)', 'ct '),
      '[^a-z0-9+]+', ' '
    )) AS protocol_text,
    upper(trim(COALESCE(p.EXAMINATION_CODE, ''))) AS protocol_code
  FROM 4_prod.bronze.map_pacs_examination p
  WHERE p.EXAMINATION_DT_TM >= TIMESTAMP'{MIN_EXAM_DATE}'
    AND p.EXAMINATION_DT_TM < TIMESTAMP'{MAX_EXAM_DATE_EXCLUSIVE}'
    AND COALESCE(p.SOURCE_PRESENT_IND, true)
    AND p.PERFORMED_EVIDENCE IS NOT NULL
    AND (
      upper(trim(COALESCE(p.MODALITY, ''))) = 'CT'
      OR lower(COALESCE(p.EXAMINATION_DESCRIPTION, '')) RLIKE r'^\s*c[.]?\s*t'
      OR upper(trim(p.EXAMINATION_CODE)) IN
         ('CCHES','CCHESC','CCHEC','CCABD','CCABDC','CCHAP','CCHAPC','CCAPC',
          'CABPE','CABPEC','CABPC','CTAPIC')
    )
),
pacs_family AS (
  SELECT *,
    CASE WHEN trim(protocol_text) = '' THEN
      CASE
        WHEN protocol_code IN ('CCHES','CCHESC','CCHEC') THEN 'CHEST'
        WHEN protocol_code IN ('CCABD','CCABDC') THEN 'CA'
        WHEN protocol_code IN ('CCHAP','CCHAPC','CCAPC','CTAPIC') THEN 'CAP'
        WHEN protocol_code IN ('CABPE','CABPEC','CABPC') THEN 'AP'
      END
    ELSE {family_sql("protocol_text")} END AS protocol_family,
    regexp_replace(protocol_text,
      r'with (and )?without|w wo|pre and post', 'with') AS contrast_text
  FROM pacs_normalized
),
pacs_scoped AS (
  SELECT *,
    (
      contrast_text RLIKE r'{NO_IV_PATTERN}'
      OR (contrast_text RLIKE r'(^| )(oral|rectal)( |$)'
          AND NOT (contrast_text RLIKE r'{IV_PATTERN}'))
    ) AS protocol_no_iv,
    CASE
      WHEN contrast_text RLIKE r'{IV_PATTERN}' THEN 'PACS_EXPLICIT_IV_PROTOCOL'
      WHEN contrast_text RLIKE r'{CONTRAST_PATTERN}' THEN 'PACS_CONTRAST_PROTOCOL'
      WHEN protocol_code IN ('CCHESC','CCHEC','CCABDC','CCHAPC','CCAPC',
                             'CABPEC','CABPC','CTAPIC') THEN 'PACS_CONTRAST_CODE'
    END AS protocol_contrast_evidence
  FROM pacs_family
  WHERE protocol_family IS NOT NULL
),
radiology_event AS (
  SELECT EVENT_ID, PERSON_ID AS MILL_PERSON_ID, ENCNTR_ID, REFERENCE_NBR,
    LEFT(REFERENCE_NBR, 16) AS ACCESSION_NBR, EXAM_TYPE_CODE,
    COALESCE(EVENT_END_DT_TM_CLEAN, EVENT_START_DT_TM_CLEAN) AS MILL_EVENT_DT_TM,
    EVENT_CD, EVENT_DESC,
    CASE EVENT_CD {EVENT_FAMILY_CASE} END AS event_family,
    CASE EVENT_CD {EVENT_CONTRAST_CASE} END AS event_contrast,
    VALID_UNTIL_DT_TM, UPDT_CNT
  FROM 4_prod.bronze.map_radiology_event
  WHERE EVENT_CD IN ({EVENT_CD_SQL})
    AND NOT COALESCE(IN_ERROR_IND, false)
),
blob_ranked AS (
  SELECT b.EVENT_ID, b.BLOB_TEXT, b.anon_text, b.TEXT_LENGTH,
    b.VERIFIED_PRSNL_ID, b.VERIFIED_DT_TM, b.PERFORMED_PRSNL_ID,
    b.ENCODING, b.parser_version, b.anon_status, b.anon_redactor_version,
    b.anon_redaction_count, b.STATUS,
    ROW_NUMBER() OVER (
      PARTITION BY b.EVENT_ID
      ORDER BY
        CASE WHEN length(trim(b.BLOB_TEXT)) > 0 THEN 0 ELSE 1 END ASC NULLS LAST,
        CASE WHEN b.VALID_UNTIL_DT_TM > current_timestamp() THEN 0 ELSE 1 END ASC NULLS LAST,
        b.VALID_FROM_DT_TM DESC NULLS LAST,
        b.UPDT_CNT DESC NULLS LAST,
        b.parser_version DESC NULLS LAST,
        b.BLOB_VERSION_ID DESC NULLS LAST
    ) AS blob_rn
  FROM 4_prod.bronze.mill_blob_text b
  WHERE b.EVENT_CD IN ({EVENT_CD_SQL})
    AND b.EVENT_ID IN (
      SELECT r.EVENT_ID
      FROM radiology_event r
      INNER JOIN pacs_scoped p
        ON p.REQUEST_ID_STRING = r.ACCESSION_NBR
       AND p.protocol_family = r.event_family
    )
),
report_candidates AS (
  SELECT p.*,
    r.EVENT_ID, r.MILL_PERSON_ID, r.ENCNTR_ID, r.REFERENCE_NBR,
    r.EXAM_TYPE_CODE, r.MILL_EVENT_DT_TM, r.EVENT_CD, r.EVENT_DESC,
    r.event_contrast,
    b.BLOB_TEXT, b.anon_text, b.TEXT_LENGTH,
    b.VERIFIED_PRSNL_ID, b.VERIFIED_DT_TM, b.PERFORMED_PRSNL_ID,
    b.ENCODING, b.parser_version, b.anon_status, b.anon_redactor_version,
    b.anon_redaction_count, b.STATUS,
    ROW_NUMBER() OVER (
      PARTITION BY p.PACS_EXAMINATION_ID
      ORDER BY
        CASE WHEN b.STATUS = 'Decoded' AND length(trim(b.BLOB_TEXT)) > 0 THEN 0 ELSE 1 END ASC NULLS LAST,
        CASE WHEN upper(trim(r.EXAM_TYPE_CODE)) = p.protocol_code THEN 0 ELSE 1 END ASC NULLS LAST,
        CASE WHEN r.VALID_UNTIL_DT_TM > current_timestamp() THEN 0 ELSE 1 END ASC NULLS LAST,
        abs(datediff(r.MILL_EVENT_DT_TM, p.EXAMINATION_DT_TM)) ASC NULLS LAST,
        b.VERIFIED_DT_TM DESC NULLS LAST,
        r.UPDT_CNT DESC NULLS LAST,
        r.EVENT_ID DESC NULLS LAST
    ) AS report_rn
  FROM pacs_scoped p
  LEFT JOIN radiology_event r
    ON p.REQUEST_ID_STRING = r.ACCESSION_NBR
   AND p.protocol_family = r.event_family
   AND (p.PERSON_ID IS NULL OR r.MILL_PERSON_ID = p.PERSON_ID)
  LEFT JOIN blob_ranked b ON b.EVENT_ID = r.EVENT_ID AND b.blob_rn = 1
),
report_selected AS (
  SELECT *,
    lower(regexp_extract(COALESCE(BLOB_TEXT, ''), r'{TECHNIQUE_PATTERN}', 1)) AS technique_line
  FROM report_candidates WHERE report_rn = 1
),
contrast_evidence AS (
  SELECT *,
    trim(regexp_replace(technique_line, '[^a-z0-9+]+', ' ')) AS technique_text
  FROM report_selected
),
eligibility AS (
  SELECT *,
    CASE
      WHEN protocol_no_iv THEN NULL
      WHEN technique_text RLIKE r'{NO_IV_PATTERN}'
        AND NOT (technique_text RLIKE r'(^| )(head|brain|neck|spine)( |$)')
        AND NOT (technique_text RLIKE r'with (and )?without|w wo|pre and post') THEN NULL
      WHEN technique_text RLIKE r'(^| )(recommend|suggest|plan|planned|consider|requested|previous|prior|comparison)( |$)'
        THEN protocol_contrast_evidence
      WHEN technique_text RLIKE r'{IV_CONTRAST_TECHNIQUE_PATTERN}'
        THEN 'REPORT_TECHNIQUE_EXPLICIT_IV'
      ELSE COALESCE(protocol_contrast_evidence,
        CASE WHEN event_contrast THEN 'MILL_CONTRAST_PROTOCOL' END)
    END AS CONTRAST_EVIDENCE
  FROM contrast_evidence
),
cohort AS (
  SELECT * EXCEPT (PERSON_ID),
    COALESCE(PERSON_ID, MILL_PERSON_ID) AS PERSON_ID,
    NULLIF(trim(REQUEST_ID_STRING), '') AS ACCESSION_NBR,
    EXAMINATION_CODE AS PACS_EXAMINATION_CODE,
    COUNT(*) OVER (PARTITION BY REQUEST_ID_STRING) AS accession_exam_count
  FROM eligibility
  WHERE CONTRAST_EVIDENCE IS NOT NULL
)
"""

print(f"Project target: {TARGET}")
print(f"Complete performed CT contrast cohort: {MIN_EXAM_DATE} <= exam < {MAX_EXAM_DATE_EXCLUSIVE}")
print("Families: CHEST, CA, CAP, AP. Contrast evidence is retained; routine contrast protocols are an IV proxy.")

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {TARGET}
COMMENT 'DAC043 CT contrast examination cohort, 2022-2024'
""")

# No pre-build DROP: CREATE OR REPLACE preserves Delta history.
# A pre-existing legacy view must be replaced by the operator before this notebook runs.
for table_name in ("ct_report_tre", "ct_report_qmul"):
    existing = spark.sql(f"""
      SELECT table_type FROM {TARGET_CATALOG}.information_schema.tables
      WHERE table_schema = '{project_identifier}' AND table_name = '{table_name}'
    """).collect()
    assert not existing or existing[0]["table_type"] != "VIEW", (
        f"{TARGET}.{table_name} is a legacy view, expected a managed Delta table."
    )

# Count before publication so downstream joins cannot silently lose or multiply exams.
expected = spark.sql(f"""
WITH {COHORT_CTES}
SELECT COUNT(*) n, COUNT(DISTINCT PACS_EXAMINATION_ID) exams
FROM cohort
""").collect()[0]
assert expected["n"] > 0, "No qualifying examinations; inspect the source before publishing."
assert expected["n"] == expected["exams"], "Cohort is not one row per PACS examination."

In [0]:
# Full TRE table. Report/identifier availability never removes a qualifying exam.
spark.sql(f"""
CREATE OR REPLACE TABLE {TARGET}.ct_report_tre AS
WITH {COHORT_CTES},
rx_ranked AS (
  SELECT
    c.PACS_EXAMINATION_ID,
    x.ExaminationStationName,
    x.ExaminationInstitution,
    x.ExaminationScheduledDate,
    x.ExaminationStat,
    ROW_NUMBER() OVER (
      PARTITION BY c.PACS_EXAMINATION_ID
      ORDER BY CASE WHEN x.ExaminationStationName IS NOT NULL THEN 0 ELSE 1 END ASC NULLS LAST,
        x.ADC_UPDT DESC NULLS LAST, x.ExaminationId DESC NULLS LAST
    ) AS rn
  FROM cohort c
  INNER JOIN 4_prod.raw.pacs_examinations x
    ON x.ExaminationId = c.PACS_EXAMINATION_ID
),
rx AS (
  SELECT *
  FROM rx_ranked
  WHERE rn = 1
),
request_ranked AS (
  SELECT
    c.PACS_EXAMINATION_ID,
    r.RequestReferringPhysician,
    r.RequestReferringUnit,
    r.RequestQuestion,
    r.RequestAnamnesis,
    ROW_NUMBER() OVER (
      PARTITION BY c.PACS_EXAMINATION_ID
      ORDER BY CASE WHEN r.RequestQuestion IS NOT NULL OR r.RequestAnamnesis IS NOT NULL THEN 0 ELSE 1 END ASC NULLS LAST,
        r.ADC_UPDT DESC NULLS LAST, r.RequestId DESC NULLS LAST
    ) AS rn
  FROM cohort c
  INNER JOIN 4_prod.raw.pacs_requests r
    ON r.RequestIdString = c.ACCESSION_NBR
),
request AS (
  SELECT *
  FROM request_ranked
  WHERE rn = 1
),
encounter_ranked AS (
  SELECT
    c.PACS_EXAMINATION_ID,
    e.encntr_type_class_desc,
    e.encntr_type_desc,
    e.med_service_desc,
    ROW_NUMBER() OVER (
      PARTITION BY c.PACS_EXAMINATION_ID
      ORDER BY
        CASE WHEN e.encntr_type_class_desc IS NOT NULL THEN 0 ELSE 1 END ASC NULLS LAST,
        e.ADC_UPDT DESC NULLS LAST
    ) AS rn
  FROM cohort c
  INNER JOIN 4_prod.bronze.map_encounter e
    ON e.ENCNTR_ID = c.ENCNTR_ID
),
encounter AS (
  SELECT *
  FROM encounter_ranked
  WHERE rn = 1
),
identifier_ranked AS (
  SELECT
    i.PERSON_ID,
    i.ALIAS_TYPE,
    i.ALIAS_VALUE,
    ROW_NUMBER() OVER (
      PARTITION BY i.PERSON_ID, i.ALIAS_TYPE
      ORDER BY
        CASE WHEN i.ALIAS_VALUE IS NOT NULL AND length(trim(i.ALIAS_VALUE)) > 0 THEN 0 ELSE 1 END ASC NULLS LAST,
        CASE WHEN i.CURRENT_IND THEN 0 ELSE 1 END ASC NULLS LAST,
        CASE WHEN i.ACTIVE_IND = 1 THEN 0 ELSE 1 END ASC NULLS LAST,
        CASE
          WHEN i.END_EFFECTIVE_DT_TM_CLEAN IS NULL
            OR i.END_EFFECTIVE_DT_TM_CLEAN > current_timestamp() THEN 0
          ELSE 1
        END ASC NULLS LAST,
        i.BEG_EFFECTIVE_DT_TM_CLEAN DESC NULLS LAST,
        i.UPDT_CNT DESC NULLS LAST,
        lower(i.ALIAS_VALUE) ASC NULLS LAST,
        CASE WHEN i.ALIAS_VALUE = upper(i.ALIAS_VALUE) THEN 0 ELSE 1 END ASC NULLS LAST,
        i.ALIAS_VALUE ASC NULLS LAST
    ) AS rn
  FROM 4_prod.bronze.map_patient_identifier i
  INNER JOIN (
    SELECT DISTINCT PERSON_ID
    FROM cohort
  ) c
    ON c.PERSON_ID = i.PERSON_ID
  WHERE i.ALIAS_TYPE IN ('MRN', 'NHS')
),
identifiers AS (
  SELECT
    PERSON_ID,
    MAX(CASE WHEN ALIAS_TYPE = 'MRN' THEN ALIAS_VALUE END) AS MRN,
    MAX(CASE WHEN ALIAS_TYPE = 'NHS' THEN ALIAS_VALUE END) AS NHS_NUMBER
  FROM identifier_ranked
  WHERE rn = 1
  GROUP BY PERSON_ID
),
accession_order AS (
  SELECT
    LEFT(r.REFERENCE_NBR, 16) AS ACCESSION_NBR,
    CASE WHEN COUNT(DISTINCT r.ORDER_ID) = 1 THEN MAX(r.ORDER_ID) END AS ORDER_ID,
    COUNT(DISTINCT r.ORDER_ID) AS ORDER_ID_CANDIDATES
  FROM 4_prod.bronze.map_radiology_event r
  WHERE r.ORDER_ID IS NOT NULL
    AND NOT COALESCE(r.IN_ERROR_IND, false)
    AND rlike(LEFT(r.REFERENCE_NBR, 16), '^UK[A-Z]{{3}}[0-9]{{11}}$')
  GROUP BY LEFT(r.REFERENCE_NBR, 16)
),
assembled AS (
  SELECT
    substr(
      sha2(concat('{PSEUDONYM_SALT}',
        CASE WHEN c.ACCESSION_NBR IS NOT NULL AND c.accession_exam_count = 1
          THEN concat(':ACC:', c.ACCESSION_NBR)
          ELSE concat(':PACS_EXAM:', CAST(c.PACS_EXAMINATION_ID AS STRING))
        END), 256),
      1,
      32
    ) AS PSEUDO_ID,
    substr(
      sha2(concat('{PSEUDONYM_SALT}',
        CASE WHEN c.PERSON_ID IS NOT NULL THEN concat(':PER:', CAST(c.PERSON_ID AS STRING))
             WHEN c.PACS_PATIENT_ID IS NOT NULL THEN concat(':PACS_PER:', CAST(c.PACS_PATIENT_ID AS STRING))
             ELSE concat(':UNKNOWN_EXAM:', CAST(c.PACS_EXAMINATION_ID AS STRING)) END), 256),
      1,
      32
    ) AS PSEUDO_PATIENT_ID,
    c.ACCESSION_NBR,
    c.STUDY_INSTANCE_UID,
    c.EVENT_ID,
    c.PERSON_ID,
    c.ENCNTR_ID,
    c.PACS_EXAMINATION_ID,
    ids.MRN,
    ids.NHS_NUMBER,
    c.EXAMINATION_DT_TM,
    c.BLOB_TEXT AS REPORT_TEXT,
    c.anon_text AS REPORT_TEXT_ANON,
    c.anon_status AS ANON_STATUS,
    c.anon_redactor_version AS ANON_REDACTOR_VERSION,
    c.anon_redaction_count AS ANON_REDACTION_COUNT,
    COALESCE(c.EXAMINATION_DESCRIPTION, c.EVENT_DESC) AS EXAMINATION_NAME,
    c.EXAMINATION_DESCRIPTION AS EXAMINATION_DESCRIPTION_PACS,
    COALESCE(c.PACS_EXAMINATION_CODE, c.EXAM_TYPE_CODE) AS EXAMINATION_CODE,
    person.gender_display AS SEX,
    CASE WHEN person.birth_date IS NOT NULL AND person.birth_date <= c.EXAMINATION_DT_TM THEN
      LEAST(CAST(floor(datediff(c.EXAMINATION_DT_TM, person.birth_date) / 365.25) AS INT), 90)
    END AS AGE_AT_EXAM,
    COALESCE(c.CLINICAL_ANAMNESIS, request.RequestAnamnesis) AS CLINICAL_DETAILS,
    COALESCE(c.CLINICAL_QUESTION, request.RequestQuestion) AS REASON_FOR_REFERRAL,
    request.RequestReferringPhysician AS REFERRING_CLINICIAN,
    COALESCE(rx.ExaminationInstitution, c.INSTITUTION) AS PERFORMING_SITE,
    COALESCE(request.RequestReferringUnit, c.REFERRING_UNIT) AS DEPARTMENT,
    encounter.med_service_desc AS MED_SERVICE,
    rx.ExaminationStationName AS ROOM,
    c.VERIFIED_DT_TM AS AUTHORISATION_DT_TM,
    orders.ORIG_ORDER_DT_TM_CLEAN AS REQUEST_DT_TM,
    CASE
      WHEN orders.ORIG_ORDER_DT_TM_CLEAN IS NOT NULL THEN 'MILL_ORDER_VIA_ACCESSION'
      ELSE 'UNAVAILABLE'
    END AS REQUEST_DT_SOURCE,
    rx.ExaminationStat AS PRIORITY_CD,
    rx.ExaminationScheduledDate AS APPOINTMENT_DT_TM,
    encounter.encntr_type_class_desc AS REQUEST_CATEGORY,
    encounter.encntr_type_desc AS ENCOUNTER_TYPE,
    verifier.NAME_FULL_FORMATTED AS AUTHORISING_RADIOLOGIST,
    c.VERIFIED_PRSNL_ID AS AUTHORISING_RADIOLOGIST_ID,
    performer.NAME_FULL_FORMATTED AS PERFORMED_RADIOGRAPHER,
    'CHEST_CA_CAP_AP' AS EXAM_FAMILY_SET,
    c.protocol_family AS EXAM_FAMILY,
    true AS CONTRAST_IND,
    c.CONTRAST_EVIDENCE,
    'PACS_EXAMINATION_DT_TM' AS EXAM_DATE_SOURCE,
    c.PERFORMED_EVIDENCE,
    CASE
      WHEN c.EVENT_ID IS NULL THEN 'NO_MATCHED_MILL_EVENT'
      WHEN c.BLOB_TEXT IS NULL OR length(trim(c.BLOB_TEXT)) = 0 THEN 'NO_REPORT_TEXT'
      WHEN c.STATUS <> 'Decoded' THEN 'REPORT_NOT_DECODED'
      WHEN c.anon_text IS NULL OR length(trim(c.anon_text)) = 0
        OR NOT (c.anon_status <=> 'anonymized') THEN 'ANON_TEXT_UNAVAILABLE'
      ELSE 'REPORT_AND_ANON_AVAILABLE'
    END AS REPORT_AVAILABILITY,
    c.MODALITY,
    c.BODY_PART,
    c.NICIP_SNOMED_CODE,
    c.IMAGE_COUNT,
    c.SERIES_COUNT_MEASURED,
    c.STORED_SIZE_BYTES,
    length(c.BLOB_TEXT) AS REPORT_CHAR_LENGTH,
    length(c.anon_text) AS ANON_REPORT_CHAR_LENGTH,
    CASE
      WHEN c.ENCODING IN ('ISO-8859-1', 'Windows-1252', 'latin-1', 'cp775')
        AND c.parser_version IS NULL THEN 'LEGACY_SINGLE_BYTE_RTF_RISK'
      WHEN c.ENCODING = 'ascii' THEN 'OK_ASCII'
      ELSE 'OK'
    END AS TEXT_QUALITY_FLAG,
    person.deceased_display AS PATIENT_DECEASED,
    CASE
      WHEN person.deceased_dt_tm IS NOT NULL
        THEN datediff(person.deceased_dt_tm, c.EXAMINATION_DT_TM)
    END AS DAYS_EXAM_TO_DEATH,
    ROW_NUMBER() OVER (
      PARTITION BY COALESCE(concat('MILL:', CAST(c.PERSON_ID AS STRING)),
        concat('PACS:', CAST(c.PACS_PATIENT_ID AS STRING)),
        concat('EXAM:', CAST(c.PACS_EXAMINATION_ID AS STRING)))
      ORDER BY c.EXAMINATION_DT_TM ASC NULLS LAST, c.PACS_EXAMINATION_ID ASC NULLS LAST
    ) AS SCAN_SEQ_FOR_PATIENT,
    datediff(
      c.EXAMINATION_DT_TM,
      LAG(c.EXAMINATION_DT_TM) OVER (
        PARTITION BY COALESCE(concat('MILL:', CAST(c.PERSON_ID AS STRING)),
        concat('PACS:', CAST(c.PACS_PATIENT_ID AS STRING)),
        concat('EXAM:', CAST(c.PACS_EXAMINATION_ID AS STRING)))
        ORDER BY c.EXAMINATION_DT_TM ASC NULLS LAST, c.PACS_EXAMINATION_ID ASC NULLS LAST
      )
    ) AS DAYS_SINCE_PREV_CAP_SCAN,
    '{PIPELINE_VERSION}' AS PIPELINE_VERSION
  FROM cohort c
  LEFT JOIN rx
    ON rx.PACS_EXAMINATION_ID = c.PACS_EXAMINATION_ID
  LEFT JOIN request
    ON request.PACS_EXAMINATION_ID = c.PACS_EXAMINATION_ID
  LEFT JOIN encounter
    ON encounter.PACS_EXAMINATION_ID = c.PACS_EXAMINATION_ID
  LEFT JOIN identifiers ids
    ON ids.PERSON_ID = c.PERSON_ID
  LEFT JOIN 4_prod.bronze.map_person person
    ON person.person_id = c.PERSON_ID
  LEFT JOIN 4_prod.bronze.map_medical_personnel verifier
    ON verifier.PERSON_ID = c.VERIFIED_PRSNL_ID
  LEFT JOIN 4_prod.bronze.map_medical_personnel performer
    ON performer.PERSON_ID = c.PERFORMED_PRSNL_ID
  LEFT JOIN accession_order accession_order
    ON accession_order.ACCESSION_NBR = c.ACCESSION_NBR
  LEFT JOIN 4_prod.bronze.map_orders orders
    ON orders.ORDER_ID = accession_order.ORDER_ID
),
priority_stats AS (
  SELECT
    PRIORITY_CD,
    COUNT(*) AS n,
    percentile_approx(
      CAST(AUTHORISATION_DT_TM AS DOUBLE) - CAST(EXAMINATION_DT_TM AS DOUBLE),
      0.5
    ) / 3600.0 AS median_turnaround_hours
  FROM assembled
  WHERE AUTHORISATION_DT_TM IS NOT NULL
    AND AUTHORISATION_DT_TM > EXAMINATION_DT_TM
  GROUP BY PRIORITY_CD
)
SELECT
  a.*,
  CASE
    WHEN a.PRIORITY_CD IS NULL OR p.n IS NULL THEN 'UNRECORDED'
    WHEN p.median_turnaround_hours <= 6 THEN 'FAST'
    WHEN p.median_turnaround_hours <= 48 THEN 'MEDIUM'
    ELSE 'SLOW'
  END AS TURNAROUND_BAND
FROM assembled a
LEFT JOIN priority_stats p
  ON a.PRIORITY_CD <=> p.PRIORITY_CD
""")

spark.sql(f"""
COMMENT ON TABLE {TARGET}.ct_report_tre IS
'DAC043 full identifiable CT contrast examination cohort, 2022-2024. One row per performed PACS examination; missing reports retained. Keep within the TRE.'
""")

In [0]:
# Queen Mary candidate view: approved anon text unchanged, no direct identifier
# columns and no DATE/TIMESTAMP columns.
spark.sql(f"""
CREATE OR REPLACE TABLE {TARGET}.ct_report_qmul AS
SELECT
  PSEUDO_ID,
  PSEUDO_PATIENT_ID,
  REPORT_TEXT_ANON AS ANON_REPORT_TEXT,
  ANON_STATUS,
  ANON_REDACTOR_VERSION,
  ANON_REDACTION_COUNT,
  ANON_REPORT_CHAR_LENGTH AS REPORT_CHAR_LENGTH,
  EXAMINATION_NAME,
  EXAMINATION_CODE,
  EXAM_FAMILY,
  CONTRAST_IND,
  CONTRAST_EVIDENCE,
  REPORT_AVAILABILITY,
  MODALITY,
  BODY_PART,
  NICIP_SNOMED_CODE,
  AGE_AT_EXAM,
  SEX,
  REQUEST_CATEGORY,
  PRIORITY_CD,
  TURNAROUND_BAND,
  YEAR(EXAMINATION_DT_TM) AS EXAM_YEAR,
  datediff(EXAMINATION_DT_TM, REQUEST_DT_TM) AS DAYS_REQUEST_TO_EXAM,
  datediff(AUTHORISATION_DT_TM, EXAMINATION_DT_TM) AS DAYS_EXAM_TO_AUTHORISATION,
  datediff(EXAMINATION_DT_TM, APPOINTMENT_DT_TM) AS DAYS_APPOINTMENT_TO_EXAM,
  SCAN_SEQ_FOR_PATIENT,
  DAYS_SINCE_PREV_CAP_SCAN,
  TEXT_QUALITY_FLAG,
  PIPELINE_VERSION
FROM {TARGET}.ct_report_tre
""")

spark.sql(f"""
COMMENT ON TABLE {TARGET}.ct_report_qmul IS
'DAC043 Queen Mary candidate view. Approved mill_blob_text.anon_text is used unchanged. No direct identifier columns or absolute structured dates. IG sign-off is still required.'
""")

In [0]:
# Tag both materialised outputs.
IG_TRE = {
    "PSEUDO_ID": ("0", "1"), "PSEUDO_PATIENT_ID": ("0", "1"),
    "ACCESSION_NBR": ("4", "2"), "STUDY_INSTANCE_UID": ("0", "1"),
    "EVENT_ID": ("0", "0"), "PERSON_ID": ("4", "2"), "ENCNTR_ID": ("4", "2"),
    "PACS_EXAMINATION_ID": ("0", "0"), "MRN": ("4", "2"), "NHS_NUMBER": ("4", "2"),
    "EXAMINATION_DT_TM": ("1", "1"), "REPORT_TEXT": ("4", "3"),
    "REPORT_TEXT_ANON": ("3", "2"), "ANON_STATUS": ("0", "0"),
    "ANON_REDACTOR_VERSION": ("0", "0"), "ANON_REDACTION_COUNT": ("0", "0"),
    "EXAMINATION_NAME": ("0", "0"), "EXAMINATION_DESCRIPTION_PACS": ("0", "0"),
    "EXAMINATION_CODE": ("0", "0"), "SEX": ("1", "1"), "AGE_AT_EXAM": ("1", "1"),
    "CLINICAL_DETAILS": ("3", "2"), "REASON_FOR_REFERRAL": ("3", "2"),
    "REFERRING_CLINICIAN": ("4", "3"), "PERFORMING_SITE": ("0", "0"),
    "DEPARTMENT": ("1", "0"), "MED_SERVICE": ("0", "0"), "ROOM": ("0", "0"),
    "AUTHORISATION_DT_TM": ("1", "1"), "REQUEST_DT_TM": ("1", "1"),
    "REQUEST_DT_SOURCE": ("0", "0"), "PRIORITY_CD": ("0", "0"),
    "APPOINTMENT_DT_TM": ("1", "1"), "REQUEST_CATEGORY": ("0", "0"),
    "ENCOUNTER_TYPE": ("0", "0"), "AUTHORISING_RADIOLOGIST": ("4", "3"),
    "AUTHORISING_RADIOLOGIST_ID": ("1", "1"), "PERFORMED_RADIOGRAPHER": ("4", "3"),
    "EXAM_FAMILY_SET": ("0", "0"), "EXAM_FAMILY": ("0", "0"),
    "CONTRAST_IND": ("0", "0"), "CONTRAST_EVIDENCE": ("0", "0"),
    "REPORT_AVAILABILITY": ("0", "0"), "MODALITY": ("0", "0"), "BODY_PART": ("0", "0"),
    "NICIP_SNOMED_CODE": ("0", "0"), "IMAGE_COUNT": ("0", "0"),
    "SERIES_COUNT_MEASURED": ("0", "0"), "STORED_SIZE_BYTES": ("0", "0"),
    "REPORT_CHAR_LENGTH": ("0", "0"), "ANON_REPORT_CHAR_LENGTH": ("0", "0"),
    "TEXT_QUALITY_FLAG": ("0", "0"), "PATIENT_DECEASED": ("1", "2"),
    "DAYS_EXAM_TO_DEATH": ("1", "2"), "SCAN_SEQ_FOR_PATIENT": ("0", "0"),
    "DAYS_SINCE_PREV_CAP_SCAN": ("0", "0"), "PIPELINE_VERSION": ("0", "0"),
    "TURNAROUND_BAND": ("0", "0"),
    "EXAM_DATE_SOURCE": ("0", "0"), "PERFORMED_EVIDENCE": ("0", "0"),
}

IG_QMUL = {
    "PSEUDO_ID": ("0", "1"), "PSEUDO_PATIENT_ID": ("0", "1"),
    "ANON_REPORT_TEXT": ("3", "2"), "ANON_STATUS": ("0", "0"),
    "ANON_REDACTOR_VERSION": ("0", "0"), "ANON_REDACTION_COUNT": ("0", "0"),
    "REPORT_CHAR_LENGTH": ("0", "0"), "EXAMINATION_NAME": ("0", "0"),
    "EXAMINATION_CODE": ("0", "0"), "EXAM_FAMILY": ("0", "0"),
    "CONTRAST_IND": ("0", "0"), "CONTRAST_EVIDENCE": ("0", "0"),
    "REPORT_AVAILABILITY": ("0", "0"), "MODALITY": ("0", "0"), "BODY_PART": ("0", "0"),
    "NICIP_SNOMED_CODE": ("0", "0"), "AGE_AT_EXAM": ("1", "1"),
    "SEX": ("1", "1"), "REQUEST_CATEGORY": ("0", "0"), "PRIORITY_CD": ("0", "0"),
    "TURNAROUND_BAND": ("0", "0"), "EXAM_YEAR": ("1", "0"),
    "DAYS_REQUEST_TO_EXAM": ("0", "0"), "DAYS_EXAM_TO_AUTHORISATION": ("0", "0"),
    "DAYS_APPOINTMENT_TO_EXAM": ("0", "0"), "SCAN_SEQ_FOR_PATIENT": ("0", "0"),
    "DAYS_SINCE_PREV_CAP_SCAN": ("0", "0"), "TEXT_QUALITY_FLAG": ("0", "0"),
    "PIPELINE_VERSION": ("0", "0"),
}

for table_name, tags in (("ct_report_tre", IG_TRE), ("ct_report_qmul", IG_QMUL)):
    columns = [field.name for field in spark.table(f"{TARGET}.{table_name}").schema]
    missing = sorted(set(columns) - set(tags))
    assert not missing, f"Missing IG classifications for {table_name}: {missing}"
    for column_name in columns:
        risk, severity = tags[column_name]
        spark.sql(
            f"ALTER TABLE {TARGET}.{table_name} ALTER COLUMN {column_name} "
            f"SET TAGS ('ig_risk' = '{risk}', 'ig_severity' = '{severity}')"
        )

In [0]:
# Schema information view: the only non-ct object retained.
spark.sql(f"""
CREATE OR REPLACE VIEW {TARGET}.schema AS
SELECT
  c.table_name,
  c.column_name,
  c.data_type,
  COALESCE(c.comment, '') AS column_comment,
  MAX(CASE WHEN t.tag_name = 'ig_risk' THEN t.tag_value END) AS ig_risk,
  MAX(CASE WHEN t.tag_name = 'ig_severity' THEN t.tag_value END) AS ig_severity
FROM {TARGET_CATALOG}.information_schema.columns c
LEFT JOIN {TARGET_CATALOG}.information_schema.column_tags t
  ON t.schema_name = c.table_schema
 AND t.table_name = c.table_name
 AND t.column_name = c.column_name
WHERE c.table_catalog = '{TARGET_CATALOG}'
  AND c.table_schema = '{project_identifier}'
  AND c.table_name IN ('ct_report_tre', 'ct_report_qmul')
GROUP BY c.table_name, c.ordinal_position, c.column_name, c.data_type, c.comment
ORDER BY c.table_name, c.ordinal_position
""")

In [0]:
# Post-build checks: complete exam grain, bounds, paired outputs and evidence.
summary = spark.sql(f"""
SELECT COUNT(*) n, COUNT(DISTINCT PACS_EXAMINATION_ID) exams,
  COUNT(DISTINCT ACCESSION_NBR) accessions,
  COUNT(DISTINCT PSEUDO_ID) pseudonyms,
  COUNT_IF(EXAMINATION_DT_TM < TIMESTAMP'{MIN_EXAM_DATE}'
        OR EXAMINATION_DT_TM >= TIMESTAMP'{MAX_EXAM_DATE_EXCLUSIVE}'
        OR EXAMINATION_DT_TM IS NULL) bad_dates,
  COUNT_IF(NOT CONTRAST_IND OR CONTRAST_EVIDENCE IS NULL) no_contrast_evidence,
  COUNT_IF(EXAM_FAMILY NOT IN ('CHEST','CA','CAP','AP') OR EXAM_FAMILY IS NULL) bad_family,
  COUNT_IF(REPORT_AVAILABILITY <> 'REPORT_AND_ANON_AVAILABLE') incomplete_reports,
  ROUND(100.0 * COUNT_IF(STUDY_INSTANCE_UID IS NOT NULL) / COUNT(*), 2) study_uid_pct
FROM {TARGET}.ct_report_tre
""").collect()[0]
qmul_rows = spark.table(f"{TARGET}.ct_report_qmul").count()
assert summary["n"] == expected["n"], f"Expected {expected['n']} examinations, built {summary['n']}."
assert summary["exams"] == summary["n"], "Enrichment duplicated examinations."
assert summary["pseudonyms"] == summary["n"], "Duplicate/missing examination pseudonyms."
assert qmul_rows == summary["n"], "TRE and QMUL membership differs."
for key in ("bad_dates", "no_contrast_evidence", "bad_family"):
    assert summary[key] == 0, f"Cohort check failed: {key}={summary[key]}"

paired = spark.sql(f"""
SELECT COUNT(*) AS differences
FROM {TARGET}.ct_report_tre t FULL OUTER JOIN {TARGET}.ct_report_qmul q
  ON t.PSEUDO_ID = q.PSEUDO_ID
WHERE t.PSEUDO_ID IS NULL OR q.PSEUDO_ID IS NULL
   OR NOT (t.REPORT_TEXT_ANON <=> q.ANON_REPORT_TEXT)
""").collect()[0]["differences"]
assert paired == 0, "QMUL membership/text differs from approved source output."
assert not [f.name for f in spark.table(f"{TARGET}.ct_report_qmul").schema
            if f.dataType.typeName() in ("date", "timestamp", "timestamp_ntz")], "Absolute QMUL dates."
display(spark.sql(f"""
SELECT YEAR(EXAMINATION_DT_TM) EXAM_YEAR, EXAM_FAMILY, CONTRAST_EVIDENCE,
  COUNT(*) EXAMINATIONS, COUNT_IF(REPORT_AVAILABILITY <> 'REPORT_AND_ANON_AVAILABLE') INCOMPLETE_REPORTS
FROM {TARGET}.ct_report_tre GROUP BY ALL ORDER BY ALL
"""))

# Keep the original residual-identifier diagnostic; no project-specific redactor.
leakage = spark.sql(f"""
WITH source AS (
  SELECT
    lower(REPORT_TEXT_ANON) AS txt,
    lower(trim(MRN)) AS mrn,
    regexp_replace(COALESCE(NHS_NUMBER, ''), '[^0-9]', '') AS nhs,
    lower(ACCESSION_NBR) AS accession,
    person.birth_date
  FROM {TARGET}.ct_report_tre report
  LEFT JOIN 4_prod.bronze.map_person person
    ON person.person_id = report.PERSON_ID
)
SELECT
  SUM(CASE WHEN mrn IS NOT NULL AND length(mrn) >= 6 AND contains(txt, mrn) THEN 1 ELSE 0 END) AS mrn,
  SUM(CASE WHEN length(nhs) = 10 AND (
      contains(txt, nhs)
      OR contains(txt, concat(substr(nhs, 1, 3), ' ', substr(nhs, 4, 3), ' ', substr(nhs, 7, 4)))
      OR contains(txt, concat(substr(nhs, 1, 3), '-', substr(nhs, 4, 3), '-', substr(nhs, 7, 4)))
    ) THEN 1 ELSE 0 END) AS nhs,
  SUM(CASE WHEN accession IS NOT NULL AND contains(txt, accession) THEN 1 ELSE 0 END) AS accession,
  SUM(CASE WHEN birth_date IS NOT NULL AND (
      contains(txt, lower(date_format(birth_date, 'dd/MM/yyyy')))
      OR contains(txt, lower(date_format(birth_date, 'dd.MM.yyyy')))
      OR contains(txt, lower(date_format(birth_date, 'dd-MM-yyyy')))
      OR contains(txt, lower(date_format(birth_date, 'yyyy-MM-dd')))
      OR contains(txt, lower(date_format(birth_date, 'd MMM yyyy')))
    ) THEN 1 ELSE 0 END) AS dob
FROM source
""").collect()[0]


print(
    f"Created {TARGET}.ct_report_tre and {TARGET}.ct_report_qmul: "
    f"{summary['n']} examinations each; {summary['accessions']} distinct accessions; "
    f"{summary['incomplete_reports']} examinations with incomplete reports retained; "
    f"STUDY_INSTANCE_UID {summary['study_uid_pct']}%."
)
print(
    f"Approved anon_text residual check: MRN={leakage['mrn']}, NHS={leakage['nhs']}, "
    f"accession={leakage['accession']}, DOB={leakage['dob']}."
)
if builtins.sum((leakage[name] or 0) for name in ("mrn", "nhs", "accession", "dob")):
    print("Residual identifiers found in the existing approved anon_text; diagnostic only. "
          "No rows or text have been changed by this notebook.")